In [10]:
from dotenv import load_dotenv

# Load OPENAI_API_KEY (and any other secrets) from a local .env file.
load_dotenv()


True

# Notebook 1 · Flat Multi-Agent System

> **Series:** LangChain MAS Architectures · Travel Agency Use Case

---

## Architecture Overview

In a **flat MAS** all agents are peers — there is no manager, no hierarchy.
Every peer can read and write to a shared board, and any peer can challenge
any earlier decision. The conversation stops when all peers vote ready.

```
┌──────────────────────────────────────────────────────┐
│                    SHARED BOARD  (state)             │
└──────┬───────────────┬────────────────┬──────────────┘
       │               │                │
  ┌────▼────┐     ┌────▼────┐     ┌─────▼────┐
  │  Peer A │◄───►│  Peer B │◄───►│  Peer C  │
  │Dest/Time│     │Booking  │     │Activities│
  └─────────┘     └─────────┘     └──────────┘
```

## Pattern in this notebook

| LangChain concept | Role in flat MAS |
|---|---|
| `AgentState` | Holds the shared board (`shared_board: list[str]`) |
| `create_agent` (peers) | Peer agents that reason about the board |
| `@tool` + `ToolRuntime` | Each peer reads & writes board via state |
| `Command(update=...)` | Appends one board entry and updates state |
| `create_agent` (coordinator) | Thin orchestrator that cycles through peer tools |


## 1 · Setup

In [11]:
# ── Traveler request ─────────────────────────────────────────────────────────
USER_REQUEST = """\
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan""".strip()

# ── Static catalog ────────────────────────────────────────────────────────────
# Agents must only use options listed here.
DESTINATIONS = {
    "Lisbon":    {"best_period": "April–June", "style": "sunny, walkable, relaxed",
                  "notes": "great for food, viewpoints, and compact neighborhoods"},
    "Barcelona": {"best_period": "April–June", "style": "lively, artistic, seaside",
                  "notes": "strong mix of architecture, beach walks, and tapas"},
    "Prague":    {"best_period": "April–June", "style": "historic, compact, lower-cost",
                  "notes": "easy sightseeing with a classic old-town atmosphere"},
}

FLIGHTS = [
    {"destination": "Lisbon",    "code": "TP-833", "price": 180, "type": "direct",  "duration": "3h 05m"},
    {"destination": "Lisbon",    "code": "IB-310", "price": 150, "type": "1 stop",  "duration": "5h 10m"},
    {"destination": "Barcelona", "code": "VY-611", "price": 140, "type": "direct",  "duration": "1h 50m"},
    {"destination": "Barcelona", "code": "IB-220", "price": 125, "type": "1 stop",  "duration": "4h 00m"},
    {"destination": "Prague",    "code": "FR-721", "price": 110, "type": "direct",  "duration": "1h 55m"},
    {"destination": "Prague",    "code": "OS-410", "price": 145, "type": "1 stop",  "duration": "3h 45m"},
]

HOTELS = [
    {"destination": "Lisbon",    "name": "Baixa Stay",       "price_per_night": 145, "style": "central boutique hotel"},
    {"destination": "Lisbon",    "name": "River Rooms",       "price_per_night": 120, "style": "simple hotel near transit"},
    {"destination": "Barcelona", "name": "Born Hotel",        "price_per_night": 160, "style": "central design hotel"},
    {"destination": "Barcelona", "name": "Gracia Inn",        "price_per_night": 130, "style": "quiet hotel in a walkable district"},
    {"destination": "Prague",    "name": "Old Town House",    "price_per_night": 115, "style": "historic hotel near main sights"},
    {"destination": "Prague",    "name": "City Garden Hotel", "price_per_night":  95, "style": "budget-friendly hotel with tram access"},
]

ACTIVITIES = [
    {"destination": "Lisbon",    "name": "Alfama food walk",                    "tag": "food",    "price": 35},
    {"destination": "Lisbon",    "name": "Belém and river tram day",            "tag": "culture", "price": 25},
    {"destination": "Barcelona", "name": "Gothic Quarter tapas evening",        "tag": "food",    "price": 40},
    {"destination": "Barcelona", "name": "Sagrada Família and modernism route", "tag": "culture", "price": 32},
    {"destination": "Prague",    "name": "Old Town walking tour",               "tag": "culture", "price": 18},
    {"destination": "Prague",    "name": "Czech dinner and jazz night",         "tag": "food",    "price": 30},
]

# ── Catalog helpers ───────────────────────────────────────────────────────────
def flights_for(destination: str)    -> list: return [f for f in FLIGHTS    if f["destination"] == destination]
def hotels_for(destination: str)     -> list: return [h for h in HOTELS     if h["destination"] == destination]
def activities_for(destination: str) -> list: return [a for a in ACTIVITIES if a["destination"] == destination]

def catalog_text() -> str:
    lines = []
    for dest, info in DESTINATIONS.items():
        lines.append(f"Destination: {dest}")
        lines.append(f"  Best period : {info['best_period']}")
        lines.append(f"  Style       : {info['style']}")
        lines.append(f"  Notes       : {info['notes']}")
        lines.append("  Flights:")
        for f in flights_for(dest):
            lines.append(f"    - {f['code']} | {f['type']} | EUR {f['price']} | {f['duration']}")
        lines.append("  Hotels:")
        for h in hotels_for(dest):
            lines.append(f"    - {h['name']} | EUR {h['price_per_night']}/night | {h['style']}")
        lines.append("  Activities:")
        for a in activities_for(dest):
            lines.append(f"    - {a['name']} | {a['tag']} | EUR {a['price']}")
        lines.append("")
    return "\n".join(lines).strip()

CATALOG_TEXT = catalog_text()

print("USER_REQUEST:")
print(USER_REQUEST)
print("\nCatalog loaded — destinations:", list(DESTINATIONS.keys()))


USER_REQUEST:
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan

Catalog loaded — destinations: ['Lisbon', 'Barcelona', 'Prague']


## 2 · Custom State

`FlatState` extends `AgentState` with one extra field:

- `shared_board` — the list of contributions every peer can read and modify.

The field starts empty and grows as each peer tool appends to it via `Command`.


In [12]:
from langchain.agents import AgentState
import operator
from typing import Annotated

class FlatState(AgentState):
    # Each peer writes a single new entry; reducer merges multiple writes per step.
    shared_board: Annotated[list[str], operator.add]


## 3 · Peer Sub-Agents

Each peer is created with `create_agent` — no tools, just a system prompt that
defines its specialty and the flat-architecture contract:

> *"You are equal to every other peer. You may refine, disagree with, or
> replace any earlier idea on the board."*

Sub-agents do not receive a `state_schema` — they are invoked directly from
inside a tool with a single `HumanMessage` that carries the board content.


In [13]:
from langchain.agents import create_agent

# ── Peer sub-agents ────────────────────────────────────────────────────────
# No tools needed: these agents only reason and produce a text contribution.
# The system prompt encodes the flat contract (equal authority, can challenge).

destination_peer = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are a peer agent in a flat travel-agency MAS. "
        "There is no manager — you have equal authority to every other peer.\n"
        "Your specialty: destination choice and best travel period.\n"
        "You MAY challenge or refine any earlier idea on the shared board.\n"
        "Only use destinations from the catalog. "
        "Reply with one concise contribution, then end your message with "
        "either 'READY: YES' or 'READY: NO' on its own line.\n"
        "Say 'READY: NO' if you think the trip isn't ready for the traveler yet, "
        "and more work is needed.\n"
        "Say 'READY: YES' if you think the trip is ready to be presented to the traveler, "
        "and no more contributions are needed."
    ),
)

booking_peer = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are a peer agent in a flat travel-agency MAS. "
        "There is no manager — you have equal authority to every other peer.\n"
        "Your specialty: flight and hotel selection for budget and convenience.\n"
        "You MAY challenge or refine any earlier idea on the shared board.\n"
        "Only use flights and hotels from the catalog. "
        "Reply with one concise contribution, then end your message with "
        "either 'READY: YES' or 'READY: NO' on its own line.\n"
        "Say 'READY: NO' if you think the trip isn't ready for the traveler yet, "
        "and more work is needed.\n"
        "Say 'READY: YES' if you think the trip is ready to be presented to the traveler, "
        "and no more contributions are needed."
    ),
)

experience_peer = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[],
    system_prompt=(
        "You are a peer agent in a flat travel-agency MAS. "
        "There is no manager — you have equal authority to every other peer.\n"
        "Your specialty: activity mix and overall trip coherence.\n"
        "You MAY challenge or refine any earlier idea on the shared board.\n"
        "Only use activities from the catalog. "
        "Reply with one concise contribution, then end your message with "
        "either 'READY: YES' or 'READY: NO' on its own line.\n"
        "Say 'READY: NO' if you think the trip isn't ready for the traveler yet, "
        "and more work is needed.\n"
        "Say 'READY: YES' if you think the trip is ready to be presented to the traveler, "
        "and no more contributions are needed."
    ),
)

print("Peer sub-agents created.")


Peer sub-agents created.


## 4 · Peer Tools (Board Read / Write via State)

Each tool follows the same three-step pattern from the reference notebook:

1. **Read state** — pull the current board from `runtime.state["shared_board"]`.
2. **Call sub-agent** — pass the board content in a `HumanMessage`.
3. **Update state** — return `Command(update={"shared_board": updated_board, ...})`.

The `ToolRuntime` parameter is injected automatically by LangChain when the tool
runs inside an agent — it exposes `runtime.state` and `runtime.tool_call_id`.

> **Why is this still flat?**
> Every peer tool reads the *same* field from state. No peer has a private
> channel to the coordinator. The board is fully transparent to all.


In [14]:
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.types import Command

# -- Helper: parse the READY vote embedded in the peer's response --
def _parse_ready(text: str) -> bool:
    # Peers end their message with 'READY: YES' or 'READY: NO'.
    # We search the last two lines so formatting noise is ignored.
    for line in reversed(text.strip().splitlines()):
        if "READY: YES" in line.upper():
            return True
        if "READY: NO" in line.upper():
            return False
    return False  # default to not ready if the marker is missing


@tool
async def destination_peer_turn(runtime: ToolRuntime) -> str:
    '''Destination peer reads the shared board and adds its contribution.'''
    # Step 1: read the current board from state
    board = runtime.state.get("shared_board", [])
    board_text = "\n".join(board) if board else "Board is empty - this is the first turn."

    # Step 2: invoke the peer sub-agent with board context
    response = await destination_peer.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Shared board so far:\n{board_text}\n\n"
            "Add one concise contribution, then state READY: YES or READY: NO."
        ))]
    })
    update_text = response["messages"][-1].content
    ready = _parse_ready(update_text)

    # Step 3: append to the board and update state
    contribution_block = "\n".join(update_text.splitlines()[:-1]).strip()
    new_entry = f"Destination peer (ready={ready}):\n{contribution_block}"
    return Command(update={
        "shared_board": [new_entry],
        "messages": [ToolMessage(update_text, tool_call_id=runtime.tool_call_id)],
    })


@tool
async def booking_peer_turn(runtime: ToolRuntime) -> str:
    '''Booking peer reads the shared board and adds its contribution.'''
    board = runtime.state.get("shared_board", [])
    board_text = "\n".join(board) if board else "Board is empty - this is the first turn."

    response = await booking_peer.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Shared board so far:\n{board_text}\n\n"
            "Add one concise contribution, then state READY: YES or READY: NO."
        ))]
    })
    update_text = response["messages"][-1].content
    ready = _parse_ready(update_text)

    contribution_block = "\n".join(update_text.splitlines()[:-1]).strip()
    new_entry = f"Booking peer (ready={ready}):\n{contribution_block}"
    return Command(update={
        "shared_board": [new_entry],
        "messages": [ToolMessage(update_text, tool_call_id=runtime.tool_call_id)],
    })


@tool
async def experience_peer_turn(runtime: ToolRuntime) -> str:
    '''Experience peer reads the shared board and adds its contribution.'''
    board = runtime.state.get("shared_board", [])
    board_text = "\n".join(board) if board else "Board is empty - this is the first turn."

    response = await experience_peer.ainvoke({
        "messages": [HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Shared board so far:\n{board_text}\n\n"
            "Add one concise contribution, then state READY: YES or READY: NO."
        ))]
    })
    update_text = response["messages"][-1].content
    ready = _parse_ready(update_text)

    contribution_block = "\n".join(update_text.splitlines()[:-1]).strip()
    new_entry = f"Experience peer (ready={ready}):\n{contribution_block}"
    return Command(update={
        "shared_board": [new_entry],
        "messages": [ToolMessage(update_text, tool_call_id=runtime.tool_call_id)],
    })

print("Peer tools defined.")


Peer tools defined.


## 5 · Coordinator Agent

The coordinator is a thin orchestrator whose only job is to call the three
peer tools in sequence across rounds. It holds the `state_schema=FlatState`
so the `shared_board` field is live-wired to the state graph.

> **Flat property preserved:** the coordinator does **not** make any planning
> decisions. It just cycles through all peers and stops when their responses
> indicate the plan is ready. The *content* decisions belong entirely to the peers.


In [15]:
from langchain.agents import create_agent

coordinator = create_agent(
    model="openai:gpt-4.1-mini",
    tools=[destination_peer_turn, booking_peer_turn, experience_peer_turn],
    state_schema=FlatState,
    system_prompt=(
        "You are the thin orchestrator of a flat travel-agency MAS.\n"
        "You do NOT make any planning decisions yourself.\n\n"
        "Your only job:\n"
        "1. Call destination_peer_turn, booking_peer_turn, and experience_peer_turn "
        "   — in that order — to complete one round.\n"
        "2. Read the tool responses. If all three peers signalled READY: YES, stop.\n"
        "3. Otherwise run another round (up to 2 rounds total).\n"
        "4. After the final round, write a short, well structured summary of the travel plan "
        "   as agreed on the shared board."
    ),
)

print("Flat MAS coordinator created.")


Flat MAS coordinator created.


## 6 · Run

In [16]:
from langchain.messages import HumanMessage

response = await coordinator.ainvoke(
    {
        # Initial human message — the traveler's request
        "messages": [HumanMessage(content=USER_REQUEST)],
        # Seed the shared board so every peer has a starting point
        "shared_board": ["Kickoff: plan a trip that satisfies the traveler request."],
    }
)


In [17]:
from pprint import pprint

# Inspect the full response dict
pprint(response)


{'messages': [HumanMessage(content='Plan a 4-day spring trip from Rome.\nRequirements:\n- mid-range budget\n- easy flights\n- central hotel\n- mix of food and culture\n- simple daily plan', additional_kwargs={}, response_metadata={}, id='41ec0d64-bcf0-43ad-972a-53e176e313d2'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 231, 'total_tokens': 242, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_2af0012656', 'id': 'chatcmpl-DPpIk4PQRMWAmhyTqaypjBtIK1V8i', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d4917-908a-7c03-a854-53b0958082d6-0', tool_calls=[{'name': 'destination_peer_turn', 'args': 

In [18]:
# Final summary produced by the coordinator after peer consensus
print(response["messages"][-1].content)


The planned 4-day spring trip from Rome is to Lisbon, chosen for its balanced mid-range budget, ease of flight, and a centrally located hotel (Baixa Stay). The itinerary offers a mix of food and cultural experiences with a simple daily plan: arrival and settling in on Day 1; an Alfama food walk on Day 2; a cultural day exploring Belém and a river tram ride on Day 3; and a relaxed final day in the Baixa neighborhood with options for light shopping and cafe visits. This plan ensures a relaxing, enriching, and well-paced trip within mid-range budget and traveler's preferences.


## 7 · Key Takeaways

| What you saw | Why it matters |
|---|---|
| `FlatState` with `shared_board` | One transparent field all peers read and write |
| Each peer tool reads `runtime.state` | No private channels — the flat contract in code |
| `Command(update={"shared_board": ...})` | State updated after every peer turn |
| Coordinator has no planning logic | Authority stays with the peers, not the orchestrator |
